In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
# File paths
TRAIN_PATH = Path("corpus/train.jsonl")
TEST_PATH = Path("corpus/test.jsonl")
VALIDATION_PATH = Path("corpus/validation.jsonl")

# Output directory
OUTPUT_DIR = Path("corpus/bertimbau_large_binary_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"

# Labels
# 0 = non-pun
# 1 = pun
id2label = {
    0: "0",
    1: "1"
}

label2id = {
    "0": 0,
    "1": 1
}

# Hyperparameters
SEED = 40
MAX_LENGTH = 256
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

In [3]:
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [4]:
def read_jsonl(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


train_df = read_jsonl(TRAIN_PATH)
validation_df = read_jsonl(VALIDATION_PATH)
test_df = read_jsonl(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(validation_df.head())
display(test_df.head())

Train shape: (3990, 5)
Validation shape: (570, 5)
Test shape: (1140, 5)


,id,text,label,tokens,labels
0,5.792.H,Por que a mulher esotérica não conseguia engra...,1,"[Por, que, a, mulher, esotérica, não, consegui...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
1,5.733.H,Qual o sambista passou a dar presente pra todo...,1,"[Qual, o, sambista, passou, a, dar, presente, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,4.652.N,Um homem matou uma ovelha e agora foi preso . ...,0,"[Um, homem, matou, uma, ovelha, e, agora, foi,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,5.2585.N,Qual apresentador de TV vive gripado? Fausto S...,0,"[Qual, apresentador, de, TV, vive, gripado, ?,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,5.28.N,Qual é a modelo mais bela que existe? Gisele B...,0,"[Qual, é, a, modelo, mais, bela, que, existe, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


,id,text,label,tokens,labels
0,5.46.H,Por que o carteiro foi à feira? Porque tinha u...,1,"[Por, que, o, carteiro, foi, à, feira, ?, Porq...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]"
1,5.1811.H,Qual é o animal que está sempre cansado? Dorme...,1,"[Qual, é, o, animal, que, está, sempre, cansad...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
2,5.1990.N,Uma cerveja se associou a um comediante para a...,0,"[Uma, cerveja, se, associou, a, um, comediante...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,4.103.H,Qual é a única coisa que se faz sempre em nume...,1,"[Qual, é, a, única, coisa, que, se, faz, sempr...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0]"
4,4.19.H,O meu baralho de cartas fica doido quando ligo...,1,"[O, meu, baralho, de, cartas, fica, doido, qua...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


,id,text,label,tokens,labels
0,4.66.N,Eu adoro fiambre . E também queijo.,0,"[Eu, adoro, fiambre, ., E, também, queijo, .]","[0, 0, 0, 0, 0, 0, 0, 0]"
1,5.2933.N,Qual a moeda inacreditável? Euro,0,"[Qual, a, moeda, inacreditável, ?, Euro]","[0, 0, 0, 0, 0, 0]"
2,5.3868.N,Qual marca de hotéis virou presidente? Trump.,0,"[Qual, marca, de, hotéis, virou, presidente, ?...","[0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,5.3612.N,Qual é a loja que fez um estande em volta do d...,0,"[Qual, é, a, loja, que, fez, um, estande, em, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,5.1714.H,Você gosta de Katy Perry? Katy perguntou?,1,"[Você, gosta, de, Katy, Perry, ?, Katy, pergun...","[0, 0, 0, 0, 0, 0, 1, 1, 0]"


In [5]:
required_columns = {"id", "text", "label"}

missing_train_columns = required_columns - set(train_df.columns)
missing_validation_columns = required_columns - set(validation_df.columns)
missing_test_columns = required_columns - set(test_df.columns)

if missing_train_columns:
    raise ValueError(f"Missing columns in train file: {missing_train_columns}")

if missing_validation_columns:
    raise ValueError(f"Missing columns in validation file: {missing_validation_columns}")

if missing_test_columns:
    raise ValueError(f"Missing columns in test file: {missing_test_columns}")

train_df["label"] = train_df["label"].astype(int)
validation_df["label"] = validation_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

valid_labels = {0, 1}

if set(train_df["label"].unique()) - valid_labels:
    raise ValueError("Train labels must be only 0 and 1.")

if set(validation_df["label"].unique()) - valid_labels:
    raise ValueError("Validation labels must be only 0 and 1.")

if set(test_df["label"].unique()) - valid_labels:
    raise ValueError("Test labels must be only 0 and 1.")

print("Train label distribution:")
display(train_df["label"].value_counts().sort_index())

print("Validation label distribution:")
display(validation_df["label"].value_counts().sort_index())

print("Test label distribution:")
display(test_df["label"].value_counts().sort_index())

Train label distribution:


,count
label,
0,1995
1,1995


Validation label distribution:


,count
label,
0,285
1,285


Test label distribution:


,count
label,
0,570
1,570


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
class PunDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.ids = dataframe["id"].astype(str).tolist()
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[index], dtype=torch.long)
        }

In [8]:
train_dataset = PunDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

validation_dataset = PunDataset(
    dataframe=validation_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = PunDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print("Train examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

Train examples: 3990
Test examples: 1140


In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision_macro = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    precision_weighted = precision_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall_weighted = recall_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1_weighted = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

In [11]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,

    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_strategy="epoch",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    save_total_limit=1,

    report_to="none",
    fp16=torch.cuda.is_available(),

    seed=SEED,
    data_seed=SEED
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ]
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.640935,0.534279,0.747368,0.747368,0.747368,0.747368,0.747368,0.747368,0.747368
2,0.505210,0.538600,0.761404,0.762036,0.761404,0.761259,0.762036,0.761404,0.761259
3,0.380324,0.556633,0.763158,0.764333,0.763158,0.762894,0.764333,0.763158,0.762894
4,0.306146,0.719077,0.761404,0.767207,0.761404,0.760101,0.767207,0.761404,0.760101
5,0.242788,1.057408,0.763158,0.763187,0.763158,0.763151,0.763187,0.763158,0.763151
6,0.174595,1.267062,0.759649,0.764100,0.759649,0.758632,0.764100,0.759649,0.758632


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2994, training_loss=0.37499961776580504, metrics={'train_runtime': 1462.129, 'train_samples_per_second': 16.373, 'train_steps_per_second': 2.048, 'total_flos': 1.115521841903616e+16, 'train_loss': 0.37499961776580504, 'epoch': 6.0})

In [14]:
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("=== Test results ===")

for metric_name, metric_value in test_results.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0.174595,1.077807,6,0.757895,0.757898,0.757895,0.757894,0.757898,0.757895,0.757894


=== Test results ===
eval_loss: 1.0778
eval_accuracy: 0.7579
eval_precision_macro: 0.7579
eval_recall_macro: 0.7579
eval_f1_macro: 0.7579
eval_precision_weighted: 0.7579
eval_recall_weighted: 0.7579
eval_f1_weighted: 0.7579


In [15]:
predictions_output = trainer.predict(test_dataset)

y_true = predictions_output.label_ids
y_pred = np.argmax(predictions_output.predictions, axis=1)

print("=== Classification report ===")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["0", "1"],
        digits=2,
        zero_division=0
    )
)

=== Classification report ===
              precision    recall  f1-score   support

           0       0.76      0.76      0.76       570
           1       0.76      0.76      0.76       570

    accuracy                           0.76      1140
   macro avg       0.76      0.76      0.76      1140
weighted avg       0.76      0.76      0.76      1140



In [16]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

display(cm_df)

,pred_0,pred_1
true_0,433,137
true_1,139,431
